In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from grape import Graph, EmbeddingResult
import pickle

/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Notebook to generate mechanisms from embeddings

### Use node and edge embedding vectors with cosine similarity

In [2]:
KG_PATH = '../../np-kg/resources/knowledge_graphs/'
EMBED_PATH = '../embedding-evaluation/embeddings/'
DATA_PATH = '../data/'
DIR_PATH = 'mech-results/'

In [3]:
npkg = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v3.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE-HASHED_v3.0.0.tsv',
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=True,
    verbose=True
)

In [9]:
npkg.get_neighbour_node_ids_from_node_id(7832).tolist()

[21829, 542684]

In [10]:
npkg.get_neighbour_node_names_from_node_name('chebi:4806')

['pr:p32418',
 'pr:q96ri1',
 'pr:p20340',
 'pr:q8wv28',
 'chebi:29103',
 'go:0031401',
 'go:1901564',
 'pr:q8nb66',
 'chebi:28088',
 'go:0045944',
 'pr:000003549',
 'chebi:28765',
 'chebi:33839',
 'go:0009890',
 'pr:000017036',
 'pr:o95716',
 'pr:q92793',
 'go:0008559',
 'mondo:0004989',
 'hp:0002076',
 'go:0043207',
 'go:0051049',
 'mondo:0008199',
 'pr:000004443',
 'chebi:18243',
 'pr:p04626',
 'chebi:53156',
 'pr:p41134',
 'pr:p12830',
 'mondo:0005015',
 'go:0065009',
 'pr:q16637',
 'chebi:63056',
 'pr:p01266',
 'go:0051171',
 'pr:q08426',
 'chebi:26523',
 'go:0010659',
 'pr:q05682',
 'go:0032553',
 'go:0050789',
 'go:1904659',
 'chebi:85682',
 'pr:p21447',
 'pr:q6s8j3',
 'pr:p10620',
 'chebi:46195',
 'go:0032774',
 'pr:p05120',
 'go:0009719',
 'go:0005956',
 'pr:a6nfq2',
 'pr:p15382',
 'pr:q07812',
 'pr:p61313',
 'chebi:5291',
 'go:0007275',
 'go:0003016',
 'pr:p09086',
 'go:0032259',
 'chebi:23456',
 'pr:q9ugk3',
 'pr:p10997',
 'pr:p15559',
 'pr:q14697',
 'go:0001774',
 'pr:000006

In [11]:
npkg.get_node_type_name_from_node_name('chebi:4806')

['chemical']

In [13]:
x = npkg.get_node_type_names()
y = []
for item in x:
    y.append(item[0])
len(y)

1089472

In [14]:
set(y)

{'adverse_event',
 'anatomy',
 'bacteria',
 'behavior',
 'bird_gene',
 'cell',
 'cell_line',
 'chemical',
 'dev_stage',
 'dictyo_genome',
 'disease',
 'drug',
 'drug_drug_interaction',
 'ecoli_gene',
 'env',
 'fly',
 'food',
 'gene',
 'mouse_pathology',
 'napdi',
 'ncit',
 'organism',
 'other',
 'pathway',
 'phenotype',
 'plant',
 'process',
 'protein',
 'rat_genome',
 'sequence',
 'trait',
 'vaccine',
 'variant',
 'worm',
 'yeast_genome',
 'zebrafish'}

In [4]:
npkg = npkg.remove_disconnected_nodes()
npkg.enable()

In [5]:
##load node and relation labels
nodeLabelsdf = pd.read_csv(KG_PATH+'nodeLabels_CURIE_v3.0.0.tsv', sep='\t')
nodeLabelsdf.head()

,source,entity_label
0,clo:0035452,ND10852 cell
1,ensembl:ENST00000256362,VRTN-201
2,dbsnp:rs1057517106,NM_003060.4(SLC22A5):c.394-1G>T
3,ensembl:ENST00000555563,SHMT2-219
4,chebi:129885,"(2S)-2-[(4R,5S)-8-[3-(dimethylamino)prop-1-yny..."


In [6]:
nodeLabelsdf.shape

(1089472, 2)

In [7]:
##convert to dictionary
nodeLabels = nodeLabelsdf.set_index('source')['entity_label'].to_dict()
len(nodeLabels)

1089472

In [3]:
##Embedding models - TransE_l1, TransR, RotateE, DistMult, ComplEx
model_name = 'ComplEx'
with open(EMBED_PATH+'npkg_'+model_name+'_embeddings.pickle', 'rb') as f:
    embedding_dict = pickle.load(f)
embedding_result = EmbeddingResult.load(embedding_dict)
node_embeddings = embedding_result.get_node_embedding_from_index(0)
edge_type_embeddings = embedding_result.get_edge_type_embedding_from_index(0)

In [4]:
node_embeddings.head()

,0,1,2,3,4,5,6,7,8,9,...,790,791,792,793,794,795,796,797,798,799
entity,,,,,,,,,,,,,,,,,,,,,
clo:0035452,0.131785,0.107176,0.090856,0.019442,0.115173,0.039812,-0.130609,0.033417,1.415244,0.058671,...,0.085796,0.005520,0.005769,0.057505,0.010282,0.056188,-0.326470,0.105168,-0.165781,-0.026152
clo:0000374,-0.110812,0.044065,-0.105546,-0.009632,-0.045707,-0.057497,-0.020060,0.117777,-0.197030,-0.164003,...,0.064172,-0.069967,-0.058874,0.030478,-0.090247,0.051408,-0.065419,-0.070068,-0.092387,-0.104509
ensembl:ENST00000256362,-0.095144,0.015026,0.087445,-0.050680,0.258873,-0.133183,-0.014490,-0.383379,1.162944,-0.114580,...,-0.007020,-0.016638,0.166905,-0.016213,-0.040768,-0.037026,-0.005199,-0.127248,-0.138623,0.015803
so:0000673,-0.091662,-0.019643,0.018833,-0.023622,0.021407,-0.079302,0.035768,-0.112718,0.394559,0.128910,...,0.121754,0.012816,0.121005,-0.011953,0.033219,0.002081,0.070068,0.036484,-0.070520,0.004353
ncbigene:55237,0.036435,0.087571,-0.006321,0.110412,-0.088299,0.023138,-0.104091,-0.322510,1.083397,-0.188881,...,-0.155078,0.105869,0.105715,0.019090,0.033007,-0.026495,-0.072331,0.210786,0.114200,0.072676


In [9]:
def get_l2_norm_edge_embedding(
    edge_embedding: np.ndarray
) -> np.ndarray:
    assert edge_embedding.ndim == 2
    return np.sqrt(np.power(edge_embedding, 2.0).sum(axis=1, keepdims=True))

In [7]:
node_embeddings.loc['clo:0035452'].to_numpy().reshape(1, -1)

array([[ 1.31785139e-01,  1.07175671e-01,  9.08562690e-02,
         1.94417778e-02,  1.15173467e-01,  3.98122370e-02,
        -1.30608752e-01,  3.34169902e-02,  1.41524446e+00,
         5.86707629e-02, -7.58316461e-03,  2.76317913e-02,
         4.06384952e-02, -6.54635504e-02,  9.02886465e-02,
        -1.01431087e-01, -3.75588685e-02,  1.85653314e-01,
        -4.41551358e-02,  6.26543611e-02,  9.53615457e-02,
        -9.36249495e-02, -6.99724704e-02,  1.97891872e-02,
         1.39532819e-01,  3.75540145e-02, -6.81964830e-02,
         6.94645718e-02, -8.74407887e-02, -4.03959583e-03,
         1.01761371e-01,  3.84224914e-02,  3.84278819e-02,
        -1.81344435e-01, -7.55026266e-02,  8.82827267e-02,
         2.07019690e-02,  2.29455441e-01,  4.83340994e-02,
         2.45392863e-02, -3.91598269e-02,  9.06975567e-03,
        -4.79937568e-02,  8.49756040e-03,  5.20059541e-02,
        -3.12271453e-02,  3.10876817e-01,  2.90789846e-02,
         4.90376763e-02,  4.54562120e-02,  1.51953876e-0

In [10]:
def get_cosine_similarity_nodes(
    source_node_embedding: np.ndarray,
    destination_node_embedding: np.ndarray
) -> np.ndarray:
    """Return cosine similarity of the two nodes."""
    hadamard_product = source_node_embedding * destination_node_embedding
    norm = (
        get_l2_norm_edge_embedding(source_node_embedding) *
        get_l2_norm_edge_embedding(destination_node_embedding)
    )
    norm[norm < 1e-6] = 1e-6  # Prevent division by very small numbers
    return np.sum(hadamard_product, axis=1, keepdims=True) / norm

In [11]:
def get_cosine_similarity_nodes_and_edges(
    source_node_embedding: np.ndarray,
    edge_type_embedding: np.ndarray,
    target_node_embedding: np.ndarray
) -> np.ndarray:
    """Return cosine similarity of the two nodes and the edge type."""
    hadamard_product = source_node_embedding * edge_type_embedding * target_node_embedding
    norm = (
        get_l2_norm_edge_embedding(source_node_embedding) *
        get_l2_norm_edge_embedding(edge_type_embedding) *
        get_l2_norm_edge_embedding(target_node_embedding)
    )
    norm[norm < 1e-6] = 1e-6  # Prevent division by very small numbers
    return np.sum(hadamard_product, axis=1, keepdims=True) / norm

In [11]:
min_weight = npkg.get_mininum_edge_weight()
max_weight = npkg.get_maximum_edge_weight()

In [12]:
def normalize_edge_weight(edge_weight, min_weight, max_weight):
    return (edge_weight - min_weight) / (max_weight - min_weight)

### Single node hop between source and target

In [40]:
def find_top_k_paths_nodes_only(source_node, k=5, similarity_threshold=0.10, use_edge_weights=False):
    source_embedding = node_embeddings.loc[source_node].to_numpy().reshape(1, -1)
    #target_embedding = node_embeddings.loc[target_node].to_numpy().reshape(1, -1)
    intermediate_nodes = []

    neighbors = npkg.get_neighbour_node_names_from_node_name(source_node)
    print('Number of neighbors:', len(neighbors))
    for node in neighbors:
        if node == source_node:
            continue
        intermediate_embedding = node_embeddings.loc[node].to_numpy().reshape(1, -1)
        similarity = get_cosine_similarity_nodes(source_embedding, intermediate_embedding)
        if use_edge_weights:
            edge_weight = npkg.get_edge_weight_from_node_names(source_node, node)
            norm_edge_weight = normalize_edge_weight(edge_weight, min_weight, max_weight)
            weight_contribution = 1-norm_edge_weight
            similarity = similarity * weight_contribution
        if similarity > similarity_threshold:
            intermediate_nodes.append((node, similarity))
    intermediate_nodes_sorted = sorted(intermediate_nodes, key=lambda x: x[1], reverse=True)
    return intermediate_nodes_sorted[:k]

In [30]:
def find_top_k_paths_with_edge(source_node, k=5, similarity_threshold=0.25, use_edge_weights=False):
    source_embedding = node_embeddings.loc[source_node].to_numpy().reshape(1, -1)
    source_node_id = npkg.get_node_id_from_node_name(source_node)
    #target_embedding = node_embeddings.loc[target_node].to_numpy().reshape(1, -1)
    intermediate_nodes = []

    neighbors = npkg.get_neighbour_node_names_from_node_name(source_node)
    print('Number of neighbors:', len(neighbors))
    for node in neighbors:
        node_id = npkg.get_node_id_from_node_name(node)
        edge_id = npkg.get_edge_id_from_node_ids(source_node_id, node_id)
        edge_type = npkg.get_edge_type_name_from_edge_id(edge_id)
        edge_embedding = edge_type_embeddings.loc[edge_type].to_numpy().reshape(1, -1)
        intermediate_node_embedding = node_embeddings.loc[node].to_numpy().reshape(1, -1)
        similarity = get_cosine_similarity_nodes_and_edges(source_embedding, edge_embedding, intermediate_node_embedding)
        if use_edge_weights:
            edge_weight = npkg.get_edge_weight_from_edge_id(edge_id)
            norm_edge_weight = normalize_edge_weight(edge_weight, min_weight, max_weight)
            weight_contribution = 1-norm_edge_weight
            similarity = similarity * weight_contribution
        if similarity > similarity_threshold:
            intermediate_nodes.append((node, similarity))
    intermediate_nodes_sorted = sorted(intermediate_nodes, key=lambda x: x[1], reverse=True)
    return intermediate_nodes_sorted[:k]

In [34]:
##green tea-nadolol-hypertension
#np_node = 'napdi_srs_imports:camellia_sinensis'
np_node = 'chebi:4806'
drug_node = 'chebi:7444'
adr_node = 'hp:0000822'
##print node ids
np_node_id = npkg.get_node_id_from_node_name(np_node)
drug_node_id = npkg.get_node_id_from_node_name(drug_node)
adr_node_id = npkg.get_node_id_from_node_name(adr_node)
np_node_id, drug_node_id, adr_node_id

(569088, 323661, 934290)

### Testing functions

In [21]:
source_node_neighbors = npkg.get_neighbour_node_names_from_node_name(np_node)
print('Number of neighbors:', len(source_node_neighbors))
source_node_embedding = node_embeddings.loc[np_node].to_numpy().reshape(1, -1)
neighbor_sim = []
for node in source_node_neighbors:
    node_embedding = node_embeddings.loc[node].to_numpy().reshape(1, -1)
    similarity = get_cosine_similarity_nodes(source_node_embedding, node_embedding)
    neighbor_sim.append((node, nodeLabels.get(node), similarity))
neighbor_sim_sorted = sorted(neighbor_sim, key=lambda x: x[2], reverse=True)
neighbor_sim_sorted[:15]

Number of neighbors: 75


[('napdi_srs_imports:camellia_sinensis_whole',
  'camellia_sinensis_whole',
  array([[0.37511632]], dtype=float32)),
 ('go:0005355',
  'glucose transmembrane transporter activity',
  array([[0.2955648]], dtype=float32)),
 ('pr:q92039', nan, array([[0.2842532]], dtype=float32)),
 ('chebi:70255',
  '(-)-epicatechin-3-O-gallate',
  array([[0.24427615]], dtype=float32)),
 ('chebi:7444', 'nadolol', array([[0.21831372]], dtype=float32)),
 ('chebi:29238', 'tritium atom', array([[0.21202286]], dtype=float32)),
 ('po:0025131',
  'plant anatomical entity',
  array([[0.20557791]], dtype=float32)),
 ('pr:862', 'Cytochrome_P450', array([[0.20301178]], dtype=float32)),
 ('chebi:17394',
  'N(5)-ethyl-L-glutamine',
  array([[0.19913742]], dtype=float32)),
 ('chebi:77635', 'calcium phosphate', array([[0.19436425]], dtype=float32)),
 ('chebi:18152', 'myricetin', array([[0.19110481]], dtype=float32)),
 ('mondo:0000001', 'disease', array([[0.1835965]], dtype=float32)),
 ('ncbitaxon:4422', nan, array([[0.1

In [22]:
source_node_neighbors = npkg.get_neighbour_node_names_from_node_name(np_node)
source_node_embedding = node_embeddings.loc[np_node].to_numpy().reshape(1, -1)
neighbor_sim = []
for node in source_node_neighbors:
    node_embedding = node_embeddings.loc[node].to_numpy().reshape(1, -1)
    similarity = get_cosine_similarity_nodes(source_node_embedding, node_embedding)
    edge_weight = npkg.get_edge_weight_from_node_names(np_node, node)
    norm_edge_weight = normalize_edge_weight(edge_weight, min_weight, max_weight)
    weight_contribution = 1-norm_edge_weight
    similarity = similarity * weight_contribution
    neighbor_sim.append((node, nodeLabels.get(node), similarity))
neighbor_sim_sorted = sorted(neighbor_sim, key=lambda x: x[2], reverse=True)
neighbor_sim_sorted[:15]

[('napdi_srs_imports:camellia_sinensis_whole',
  'camellia_sinensis_whole',
  array([[0.3749933]], dtype=float32)),
 ('go:0005355',
  'glucose transmembrane transporter activity',
  array([[0.29546472]], dtype=float32)),
 ('pr:q92039', nan, array([[0.28415987]], dtype=float32)),
 ('chebi:70255',
  '(-)-epicatechin-3-O-gallate',
  array([[0.24419242]], dtype=float32)),
 ('chebi:7444', 'nadolol', array([[0.21823765]], dtype=float32)),
 ('chebi:29238', 'tritium atom', array([[0.21195167]], dtype=float32)),
 ('po:0025131',
  'plant anatomical entity',
  array([[0.2048125]], dtype=float32)),
 ('pr:862', 'Cytochrome_P450', array([[0.20272799]], dtype=float32)),
 ('chebi:17394',
  'N(5)-ethyl-L-glutamine',
  array([[0.19906968]], dtype=float32)),
 ('chebi:77635', 'calcium phosphate', array([[0.19429968]], dtype=float32)),
 ('chebi:18152', 'myricetin', array([[0.19103831]], dtype=float32)),
 ('mondo:0000001', 'disease', array([[0.18342414]], dtype=float32)),
 ('ncbitaxon:4422', nan, array([[0.

In [35]:
'''source_node_neighbors = npkg.get_neighbour_node_names_from_node_name(source_node)
source_node_embedding = node_embeddings.loc[source_node].to_numpy().reshape(1, -1)
neighbor_sim = []
for node in source_node_neighbors:
    node_id = npkg.get_node_id_from_node_name(node)
    node_embedding = node_embeddings.loc[node].to_numpy().reshape(1, -1)
    edge_id = npkg.get_edge_id_from_node_ids(source_node_id, node_id)
    edge_type = npkg.get_edge_type_name_from_edge_id(edge_id)
    edge_embedding = edge_type_embeddings.loc[edge_type].to_numpy().reshape(1, -1)
    similarity = get_cosine_similarity_nodes_and_edges(source_node_embedding, edge_embedding, node_embedding)
    neighbor_sim.append((node, similarity))
neighbor_sim_sorted = sorted(neighbor_sim, key=lambda x: x[1], reverse=True)
neighbor_sim_sorted[:15]'''

[('po:0025131', array([[0.00554093]], dtype=float32)),
 ('chebi:89263', array([[0.00496541]], dtype=float32)),
 ('chebi:69304', array([[0.00438001]], dtype=float32)),
 ('chebi:180713', array([[0.00379069]], dtype=float32)),
 ('chebi:4918', array([[0.00317379]], dtype=float32)),
 ('hp:0002315', array([[0.00309839]], dtype=float32)),
 ('chebi:28768', array([[0.00307508]], dtype=float32)),
 ('chebi:169400', array([[0.00290308]], dtype=float32)),
 ('chebi:31402', array([[0.00286411]], dtype=float32)),
 ('chebi:81229', array([[0.0025952]], dtype=float32)),
 ('chebi:17187', array([[0.00258067]], dtype=float32)),
 ('chebi:172669', array([[0.00252918]], dtype=float32)),
 ('go:0040007', array([[0.00187221]], dtype=float32)),
 ('chebi:4917', array([[0.00132112]], dtype=float32)),
 ('napdi_srs_imports:cinnamomum_verum_whole',
  array([[0.00128716]], dtype=float32))]

In [36]:
'''source_node_neighbors = npkg.get_neighbour_node_names_from_node_name(source_node)
source_node_embedding = node_embeddings.loc[source_node].to_numpy().reshape(1, -1)
neighbor_sim = []
for node in source_node_neighbors:
    node_id = npkg.get_node_id_from_node_name(node)
    node_embedding = node_embeddings.loc[node].to_numpy().reshape(1, -1)
    edge_id = npkg.get_edge_id_from_node_ids(source_node_id, node_id)
    edge_type = npkg.get_edge_type_name_from_edge_id(edge_id)
    edge_embedding = edge_type_embeddings.loc[edge_type].to_numpy().reshape(1, -1)
    similarity = get_cosine_similarity_nodes_and_edges(source_node_embedding, edge_embedding, node_embedding)
    edge_weight = npkg.get_edge_weight_from_edge_id(edge_id)
    norm_edge_weight = normalize_edge_weight(edge_weight, min_weight, max_weight)
    weight_contribution = 1-norm_edge_weight
    similarity = similarity * weight_contribution
    neighbor_sim.append((node, similarity))
neighbor_sim_sorted = sorted(neighbor_sim, key=lambda x: x[1], reverse=True)
neighbor_sim_sorted[:15]'''

[('po:0025131', array([[0.0055203]], dtype=float32)),
 ('chebi:89263', array([[0.00496378]], dtype=float32)),
 ('chebi:69304', array([[0.00437858]], dtype=float32)),
 ('chebi:180713', array([[0.00378945]], dtype=float32)),
 ('chebi:4918', array([[0.00317271]], dtype=float32)),
 ('hp:0002315', array([[0.00309588]], dtype=float32)),
 ('chebi:28768', array([[0.00307407]], dtype=float32)),
 ('chebi:169400', array([[0.00290213]], dtype=float32)),
 ('chebi:31402', array([[0.00286317]], dtype=float32)),
 ('chebi:81229', array([[0.00259435]], dtype=float32)),
 ('chebi:17187', array([[0.00257982]], dtype=float32)),
 ('chebi:172669', array([[0.00252835]], dtype=float32)),
 ('go:0040007', array([[0.00187138]], dtype=float32)),
 ('chebi:4917', array([[0.00132014]], dtype=float32)),
 ('napdi_srs_imports:cinnamomum_verum_whole',
  array([[0.00128674]], dtype=float32))]

In [23]:
node_embeddings.head()

,0,1,2,3,4,5,6,7,8,9,...,790,791,792,793,794,795,796,797,798,799
entity,,,,,,,,,,,,,,,,,,,,,
clo:0035452,0.131785,0.107176,0.090856,0.019442,0.115173,0.039812,-0.130609,0.033417,1.415244,0.058671,...,0.085796,0.005520,0.005769,0.057505,0.010282,0.056188,-0.326470,0.105168,-0.165781,-0.026152
clo:0000374,-0.110812,0.044065,-0.105546,-0.009632,-0.045707,-0.057497,-0.020060,0.117777,-0.197030,-0.164003,...,0.064172,-0.069967,-0.058874,0.030478,-0.090247,0.051408,-0.065419,-0.070068,-0.092387,-0.104509
ensembl:ENST00000256362,-0.095144,0.015026,0.087445,-0.050680,0.258873,-0.133183,-0.014490,-0.383379,1.162944,-0.114580,...,-0.007020,-0.016638,0.166905,-0.016213,-0.040768,-0.037026,-0.005199,-0.127248,-0.138623,0.015803
so:0000673,-0.091662,-0.019643,0.018833,-0.023622,0.021407,-0.079302,0.035768,-0.112718,0.394559,0.128910,...,0.121754,0.012816,0.121005,-0.011953,0.033219,0.002081,0.070068,0.036484,-0.070520,0.004353
ncbigene:55237,0.036435,0.087571,-0.006321,0.110412,-0.088299,0.023138,-0.104091,-0.322510,1.083397,-0.188881,...,-0.155078,0.105869,0.105715,0.019090,0.033007,-0.026495,-0.072331,0.210786,0.114200,0.072676


### Testing paths

In [43]:
top_intermediates = find_top_k_paths_nodes_only(np_node, k=20, similarity_threshold=0.10, use_edge_weights=False)
top_intermediates

Number of neighbors: 1875


[('chebi:3962', array([[0.4199009]], dtype=float32)),
 ('chebi:28088', array([[0.40285844]], dtype=float32)),
 ('chebi:29073', array([[0.39982533]], dtype=float32)),
 ('chebi:15600', array([[0.39728883]], dtype=float32)),
 ('chebi:6801', array([[0.3904836]], dtype=float32)),
 ('chebi:17963', array([[0.38567328]], dtype=float32)),
 ('chebi:15551', array([[0.3796986]], dtype=float32)),
 ('chebi:15428', array([[0.37513122]], dtype=float32)),
 ('chebi:37527', array([[0.36474618]], dtype=float32)),
 ('chebi:44616', array([[0.3638891]], dtype=float32)),
 ('chebi:27732', array([[0.36227298]], dtype=float32)),
 ('chebi:52717', array([[0.34587374]], dtype=float32)),
 ('chebi:42588', array([[0.34568995]], dtype=float32)),
 ('chebi:26333', array([[0.34162715]], dtype=float32)),
 ('chebi:75142', array([[0.34091538]], dtype=float32)),
 ('chebi:29320', array([[0.34073573]], dtype=float32)),
 ('chebi:77954', array([[0.3391985]], dtype=float32)),
 ('chebi:34317', array([[0.33909237]], dtype=float32)),

In [41]:
top_intermediates = find_top_k_paths_nodes_only(np_node, k=5, similarity_threshold=0.10, use_edge_weights=True)
top_intermediates

Number of neighbors: 1875


[('chebi:3962', array([[0.41958126]], dtype=float32)),
 ('chebi:28088', array([[0.40266556]], dtype=float32)),
 ('chebi:29073', array([[0.39959013]], dtype=float32)),
 ('chebi:15600', array([[0.39683768]], dtype=float32)),
 ('chebi:6801', array([[0.3903116]], dtype=float32))]

In [38]:
##path from np_node to drug_node with one intermediate node
for intermediate, similarity in top_intermediates:
    ##if intermediate node has edge to target node, then find top k paths with edge
    intermediate_node_id = npkg.get_node_id_from_node_name(intermediate)
    linked_node_ids_intermediate = npkg[intermediate_node_id]
    linked_node_ids_drug = npkg[drug_node_id]
    if drug_node_id in linked_node_ids_intermediate or intermediate_node_id in linked_node_ids_drug:
        print('Intermediate node:', nodeLabels.get(intermediate))
        target_embedding = node_embeddings.loc[drug_node].to_numpy().reshape(1, -1)
        final_similarity = get_cosine_similarity_nodes(
            node_embeddings.loc[intermediate].to_numpy().reshape(1, -1), target_embedding
        )
        path_score = (similarity + final_similarity) / 2
    
        print(f"Path: {nodeLabels.get(np_node)} -> {nodeLabels.get(intermediate)} -> {nodeLabels.get(drug_node)}, Score: {path_score}")

Intermediate node: (+)-catechin
Path: (-)-epigallocatechin 3-gallate -> (+)-catechin -> nadolol, Score: [[0.3118347]]


In [ ]:
'''target_node_id = npkg.get_node_id_from_node_name(target_node)
for intermediate, similarity in top_intermediates_2:
    target_embedding = node_embeddings.loc[target_node]
    intermediate_node_id = npkg.get_node_id_from_node_name(intermediate)
    edge_id = npkg.get_edge_id_from_node_ids(intermediate, target_node_id)
    edge_type = npkg.get_edge_type_name_from_edge_id(edge_id)
    edge_embedding = edge_type_embeddings.loc[edge_type]

    final_similarity = get_cosine_similarity_nodes_and_edges(
        node_embeddings.loc[intermediate], edge_embedding, target_embedding
    )
    path_score = (similarity + final_similarity) / 2
    print(f"Path: {source_node} -> {intermediate} -> {target_node}, Score: {path_score}")'''

### Variable path lengths

In [ ]:
##for variable path lengths, do recursive search for intermediate nodes -- TODO